# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring the FAIR² Mental Health Survey dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# .metadata is a dataclass-like object, not a dict
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each entity in a Croissant schema, such as record sets and fields, has a unique `@id`. We'll enumerate all record sets and sample fields for reference.

In [ ]:
# List all available record sets and their ids
record_sets = dataset.record_sets
print("Record sets:")
for rs in record_sets:
    print(f"  Name: {rs.name}, @id: {rs.id}")

# For each record set, list the fields (columns)
print("\nFields in each record set:")
for rs in record_sets:
    print(f"- Record Set: {rs.name} (@id: {rs.id})")
    for f in rs.fields:
        # Each field object has .name and .id
        print(f"    Field: {f.name}, @id: {f.id}, dtype: {f.data_type}")

For convenience, let's select one main record set id and a few fields for downstream demonstration.

In [ ]:
# Select the first record set for demonstration
main_rs = record_sets[0]
main_record_set_id = main_rs.id
print(f"Using record set: {main_rs.name} (@id: {main_record_set_id})")

# List field ids
field_ids = [f.id for f in main_rs.fields]
print("Fields:")
for f in main_rs.fields:
    print(f"  {f.name} (@id: {f.id})  - dtype: {f.data_type}")

# Find a numeric field for EDA
numeric_fields = [f.id for f in main_rs.fields if f.data_type in ("Integer", "Float", "Number")]
if numeric_fields:
    print(f"Numeric fields: {numeric_fields}")
    example_numeric_field = numeric_fields[0]
else:
    example_numeric_field = field_ids[0]  # fallback

## 3. Data Extraction
Load data from each record set into a DataFrame, using its `@id`. We'll demonstrate with the main record set selected above.

In [ ]:
# Extract data from all record sets into DataFrames
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded DataFrame for record set: {rs.name} (@id: {rs.id}) -- {len(df)} rows, columns: {list(df.columns)}")

# Preview data from the main record set
main_df = dataframes[main_record_set_id]
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data wrangling—filtering numeric fields, handling missing values, normalization, and grouping. All field and record set references use their `@id` identifiers.

In [ ]:
# Choose a numeric field and a field for grouping
numeric_field = example_numeric_field
group_field = None

# Heuristically pick a group-by field (e.g., 'level_of_education', 'gender', etc. if present)
for f in main_rs.fields:
    # choose a likely categorical field
    if f.data_type.lower() in ('string', 'text') and f.id != numeric_field:
        group_field = f.id
        break

print(f"Using numeric field: {numeric_field}")
if group_field is not None:
    print(f"Using group field: {group_field}")
else:
    print("No suitable group field was found.")

# Drop missing values for numeric field
eda_df = main_df.dropna(subset=[numeric_field])

# Filter records with numeric_field > threshold
threshold = eda_df[numeric_field].mean() if eda_df[numeric_field].dtype.kind in 'fi' else 0
filtered_df = eda_df[eda_df[numeric_field] > threshold]
print(f"Filtered records in {main_record_set_id} where {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize (z-score) the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field}, mean of {numeric_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions of selected fields. Here, we will show the distribution of the main numeric field, and if available, group means by categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field exists, plot grouped means (e.g., bar chart)
if group_field and group_field in main_df.columns:
    plt.figure(figsize=(10,4))
    grouped = main_df.groupby(group_field)[numeric_field].mean().reset_index()
    sns.barplot(data=grouped, x=group_field, y=numeric_field, palette="muted")
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook illustrated how to use `mlcroissant` to:
- Load a Croissant-defined dataset by URL
- Programmatically inspect record sets and fields using their `@id`
- Extract data for analysis with pandas
- Conduct simple EDA: filtering, normalization, and grouping
- Visualize distributions and group summaries

The [FAIR² Mental Health Survey](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) demonstrates machine-actionable metadata and AI-ready formats for public health research in Africa.

**Next steps:** You can adapt this notebook for more advanced statistical analysis or machine learning, leveraging the standardized schema provided by Croissant.